# WeakTr: resume Phase 2 after timeout

Your supplied log contains **60/60 Phase-1 epochs**, with best single-scale selection CAM mIoU **65.84% at epoch 34**. The later run completed **29/100 Phase-2 epochs**, reaching **69.53% validation mIoU**, before a 12-hour timeout. This notebook now defaults to restoring the latest **active Phase-2 checkpoint and pseudo-labels**, then continuing Phase 2 in bounded sessions. It does not restart Phase 1.

Before running, enable **Internet** and a **GPU**, and attach these two private datasets:

1. Original `weaktr-kaggle-inputs.zip`, extracted to include `weaktr-inputs-manifest.json`.
2. The cancelled run's complete **active `weaktr-artifacts` folder**, including `phase1/`, `phase2/checkpoint.pth`, `phase2/variant.yml`, `pseudo/`, and logs/records. Attach the run outputs or upload this folder as a private dataset. An earlier resume ZIP can predate Phase 2 and is insufficient to recover its updates. A manifest alone cannot restore training; the old notebook is not an input dataset.

Run cells in order. The first cell clones/updates helper code; no Windows-side files are required. A Kaggle Jupyter server selected in VS Code uses the same workflow. Temporary environment/data/CAMs live in `/kaggle/tmp/weaktr-runtime`; durable results and `weaktr-resume.zip` stay in `/kaggle/working`. Scratch is not preserved between sessions.

Use interactive execution first to inspect CAM scores and storage. **Save & Run All** executes the remaining stages through a bounded Phase-2 chunk (30 additional epochs by default). Final evaluation runs only after the saved checkpoint confirms all 100 epochs. `PHASE2_PILOT = True` runs only the one-epoch pilot. The 74.0% final validation target remains unmeasured. GPU smoke checks passed in the earlier session; a new GPU/session can be checked again with `RUN_SMOKE = True`.

In [ ]:
# 1. Configuration. Defaults resume Phase 2 after the cancelled session.
from pathlib import Path
import json, os, subprocess, sys

if not Path('/kaggle').is_dir():
    raise RuntimeError('Run in Kaggle or select the Kaggle remote Jupyter kernel.')

REPO = Path('/kaggle/working/WeakTr')
WORK = Path('/kaggle/working')
INPUTS = Path('/kaggle/input')
SCRATCH = Path('/kaggle/tmp/weaktr-runtime')
RESUME_FROM = 'auto'  # Or the exact directory containing the desired run-manifest.json.
START_FROM = 'phase2'  # phase1 | cams | pseudo | phase2 | evaluate
RUN_SMOKE = False   # Optional: recheck both models in this GPU session.
PHASE2_PILOT = False  # True: one Phase-2 epoch, then stop before full training/evaluation.
PHASE2_EPOCHS_THIS_SESSION = 30  # About 10.3 training/validation hours at the observed speed.

STAGES = ('phase1', 'cams', 'pseudo', 'phase2', 'evaluate')
if START_FROM not in STAGES:
    raise ValueError(f'START_FROM must be one of {STAGES}')

if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', 'reproduction/voc-deit-small',
                    'https://github.com/ntmphu/WeakTr.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin',
                    'reproduction/voc-deit-small'], check=True)

def workflow(stage, *, skip_install=False):
    command = [sys.executable, str(REPO / 'reproduction/kaggle.py'), stage,
               '--inputs', str(INPUTS), '--work', str(WORK), '--scratch', str(SCRATCH),
               '--resume-from', RESUME_FROM]
    if stage == 'phase2':
        command += ['--epochs-this-session', str(PHASE2_EPOCHS_THIS_SESSION)]
    if skip_install:
        command.append('--skip-install')
    subprocess.run(command, cwd=REPO, check=True)

def should_run(stage):
    return STAGES.index(stage) >= STAGES.index(START_FROM)

print({'kernel': sys.executable, 'start_from': START_FROM, 'resume': RESUME_FROM,
       'temporary': str(SCRATCH), 'saved_outputs': str(WORK)})

def phase2_complete():
    script = ('import sys; from pathlib import Path; '
              'from reproduction.checkpoint import load_checkpoint; '
              'p = Path(sys.argv[1]); s = load_checkpoint(p) if p.is_file() else {}; '
              'assert not s or (s.get("stage") == "phase2" and s.get("format_version") == 1); '
              'print(s.get("epoch", -1) + 1)')
    result = subprocess.run([str(SCRATCH / 'weaktr-env/bin/python'), '-c', script,
                             str(WORK / 'weaktr-artifacts/phase2/checkpoint.pth')],
                            cwd=REPO, stdout=subprocess.PIPE, text=True, check=True)
    completed = int(result.stdout.strip())
    print(f'Phase 2 checkpoint: {completed}/100 completed epochs')
    return completed >= 100


## 2. Restore and prepare

Restores an attached saved run with hash verification and validates inputs. Setup hides CUDA and does not train. Existing local checkpoints take precedence in `auto` mode. A complete installed environment is reused automatically; missing or broken dependencies trigger normal installation. Both trainer/CRF imports, CPU tests, and real-data checks still run.

Wait for **PREPARATION PASSED**. If the old session ran out of space, an optional cell `!python -m pip cache purge` clears installer downloads without deleting checkpoints. The launcher relocates known old temporary paths and reports storage usage/free capacity. Scratch relocation reduces saved output size; it cannot create physical capacity on a shared full filesystem.

In [ ]:
# Prefer the accessible active Phase-2 folder over an older exported ZIP.
if START_FROM in ('phase2', 'evaluate') and RESUME_FROM == 'auto' and not (WORK / 'weaktr-artifacts/phase2/checkpoint.pth').is_file():
    candidates = sorted({p.parent.parent for p in INPUTS.rglob('checkpoint.pth')
                         if p.parent.name == 'phase2'
                         and (p.parent.parent / 'phase1/checkpoint_best_mIoU.pth').is_file()
                         and (p.parent.parent / 'pseudo').is_dir()})
    if len(candidates) == 1:
        RESUME_FROM = str(candidates[0])
        print('Recover latest active Phase-2 artifacts from:', RESUME_FROM)
    elif len(candidates) > 1:
        raise RuntimeError('Multiple Phase-2 runs found; set RESUME_FROM to the desired active artifact folder.')
    else:
        raise RuntimeError('Attach the cancelled run active artifact folder containing Phase-2 checkpoint and pseudo-labels. The older Phase-1 ZIP is not enough.')

python = SCRATCH / 'weaktr-env/bin/python'
reuse_environment = False
if python.is_file():
    check_env = os.environ.copy()
    check_env['CUDA_VISIBLE_DEVICES'] = ''
    check_env['PYTHONPATH'] = os.pathsep.join([str(REPO), str(REPO / 'OnlineRetraining')])
    try:
        probe = subprocess.run(
            [str(python), '-c', 'import sys; assert sys.version_info[:2] == (3, 10); import main; import segm.train; import mmcv.ops; import pydensecrf.densecrf'],
            cwd=REPO, env=check_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        reuse_environment = probe.returncode == 0
    except OSError:
        print('Existing interpreter is unusable; setup will repair it.')
print('Reuse installed environment:', reuse_environment)
workflow('prepare', skip_install=reuse_environment)

## 3. Check the completed Phase-1 snapshots

The default resume route requires the full 60-epoch latest snapshot and the best CAM snapshot. This check prints their recorded epochs and score; it fails rather than silently starting new training if the saved run is missing/incomplete. Uploaded files were hash-checked during restoration. Only use checkpoints from your trusted saved run.

In [ ]:
if START_FROM in ('phase2', 'evaluate') and not (WORK / 'weaktr-artifacts/phase2/checkpoint.pth').is_file():
    raise RuntimeError('This resume route needs the actual Phase-2 checkpoint. Recover it from the cancelled run; an earlier Phase-1 ZIP is insufficient.')

if START_FROM != 'phase1':
    check_script = r"""
import json
from pathlib import Path
from reproduction.checkpoint import load_checkpoint
root = Path('/kaggle/working/weaktr-artifacts/phase1')
latest_path = root / 'checkpoint.pth'
best_path = root / 'checkpoint_best_mIoU.pth'
if not latest_path.is_file() or not best_path.is_file():
    raise RuntimeError('Attach the extracted saved run with both Phase-1 checkpoints, then rerun setup.')
latest = load_checkpoint(latest_path)
if latest.get('stage') != 'phase1' or latest.get('format_version') != 1 or latest.get('epoch', -1) < 59:
    raise RuntimeError('The latest Phase-1 snapshot does not confirm 60 completed epochs. Restore the correct run.')
del latest
best = load_checkpoint(best_path)
if best.get('stage') != 'phase1' or best.get('format_version') != 1:
    raise RuntimeError('The best CAM checkpoint is not a full Phase-1 snapshot.')
print(json.dumps({'phase1_completed_epochs': 60, 'best_cam_epoch': best['epoch'] + 1,
                  'best_selection_cam_miou': best.get('miou', best['best']['max_max_mIoU']),
                  'checkpoint': str(best_path)}, indent=2))
"""
    subprocess.run([str(SCRATCH / 'weaktr-env/bin/python'), '-c', check_script], cwd=REPO, check=True)
else:
    print('Phase-1 training route selected explicitly.')
if START_FROM in ('phase2', 'evaluate'):
    phase2_complete()  # Prints the actual restored completed-epoch count before training.


## 4. Optional GPU smoke check

Both model paths were checked in your earlier session. Set `RUN_SMOKE = True` to repeat the synthetic checks after a GPU/environment change. This is separate from a real training epoch and from controlled recovery equivalence.

In [ ]:
if RUN_SMOKE:
    workflow('smoke')
else:
    print('Optional smoke check skipped.')

## 5. Phase 1 — skipped by default

With `START_FROM = 'phase2'` (or `'cams'`), this cell performs no training. Only explicitly selecting `START_FROM = 'phase1'` runs/resumes the official 60-epoch Phase-1 schedule. Your supplied run has already completed it.

In [ ]:
if should_run('phase1'):
    workflow('phase1')
else:
    print('Phase 1 already complete; continuing from the saved best CAM checkpoint.')

## 6. Generate multiscale CAMs

Uses `checkpoint_best_mIoU.pth`, scales 1.0/1.2, and all 10,582 training images. This does not train Phase 1. Complete saved pseudo-labels are reused when available. CAMs are temporary: if a session ends before pseudo-label generation, rerun this stage next session.

In [ ]:
if should_run('cams'):
    workflow('cams')
else:
    print('CAM generation skipped by START_FROM.')

## 7. Evaluate multiscale CAMs

Evaluates the official 1,464-image training selection list across background thresholds 0.40–0.59. Inspect **Best background score ... mIoU ...** before proceeding interactively. The small result file is `/kaggle/working/weaktr-artifacts/cam-eval.txt`. This is a CAM metric, separate from final segmentation validation mIoU. No score threshold blocks the full workflow automatically.

In [ ]:
if should_run('cams'):
    workflow('cam-eval')

## 8. Generate and preserve CRF pseudo-labels

Uses the official fixed threshold argument 43 (0.43), not the threshold selected above. Validates the complete pseudo-label set, then exports the updated resume ZIP. Download it before ending the session. These masks become Phase-2 training targets; author masks are not substituted.

In [ ]:
if should_run('pseudo'):
    workflow('pseudo')
    workflow('preserve')
else:
    print('Using previously saved pseudo-labels; Phase 2 validates their complete coverage.')

## 9. Phase 2: optional one-epoch pilot or full training

`PHASE2_PILOT = False` runs/resumes the original 100-epoch schedule: batch 4, SGD/poly LR 0.0001, encoder multiplier 0.1, resize 520, crop/window 480, stride 320, gradient clipping 1.2/120, and validation every epoch.

For a first real-data check, set `PHASE2_PILOT = True` in Configuration. This cell then stops after one completed Phase-2 epoch; later evaluation cells are skipped. To continue, set `PHASE2_PILOT = False` and `START_FROM = 'phase2'`, rerun Configuration, and rerun this training cell in the same prepared session. After a runtime restart, rerun setup and checkpoint checks first. The pilot snapshot resumes with the original full scheduler.

Each normal Phase-2 invocation is capped at **30 additional epochs** by `PHASE2_EPOCHS_THIS_SESSION`. The complete schedule and scheduler remain 100 epochs. At the speed measured in the supplied log, 30 epochs take about 10.3 hours plus setup. After a chunk completes, checkpoints and the resume ZIP are exported and final evaluation is skipped until all 100 epochs are complete. Download the ZIP, attach it in the next session, set `START_FROM = 'phase2'`, and repeat. This is an epoch cap, not a guaranteed wall-clock deadline if hardware or preprocessing time changes.

After a hard cancellation, an older ZIP can predate active training checkpoints. Recover the actual `weaktr-artifacts/phase2/` folder plus pseudo-labels from the cancelled run's files if available; the console log cannot recreate weights. Verify the checkpoint epoch before assuming training progress is preserved.

In [ ]:
if should_run('phase2'):
    workflow('phase2-pilot' if PHASE2_PILOT else 'phase2')
    if PHASE2_PILOT:
        print('Phase-2 pilot complete. Download weaktr-resume.zip. Set PHASE2_PILOT=False and START_FROM=phase2 to continue.')

## 10. Final multiscale evaluation and CRF

Skipped during the Phase-2 pilot. Preserves the official two-decoder combination (`backbone=multi`, weight 0.8). Raw per-class/mean scores are stored in `weaktr-artifacts/phase2/scores_ms.yml`; CRF scores are in the saved stage log. Record raw and post-CRF scores separately. The 74.0% reference is a target, not an assumed outcome.

In [ ]:
if should_run('evaluate') and not PHASE2_PILOT:
    if phase2_complete():
        workflow('evaluate')
        workflow('crf')
        workflow('preserve')
    else:
        print('SESSION COMPLETE. Download weaktr-resume.zip. Resume next session with START_FROM=phase2; final evaluation waits for 100 epochs.')
else:
    print('Final evaluation skipped during the Phase-2 pilot.')

## Save and resume another session

Download `/kaggle/working/weaktr-resume.zip` after completed stages and before quota expiry. It includes full training state, best checkpoints, configs, metrics, logs, and pseudo-labels. Temporary environment/data/CAM/probability collections are excluded. The active artifacts folder is also in saved notebook outputs; new exports no longer create an additional unpacked copy.

Next session: attach both original inputs and the extracted newest resume ZIP, keep `RESUME_FROM = 'auto'`, and choose `START_FROM = 'phase2'` when the complete pseudo-label set is saved, or `START_FROM = 'evaluate'` after all 100 Phase-2 epochs. Rerun setup and checkpoint checks. Existing local checkpoints win in automatic mode, so use a clean runtime to switch to a different saved run.

Checkpoints are saved at completed epoch boundaries. A hard interruption replays an unfinished epoch and cannot execute ZIP export after termination. A live session's disk alone is not a durable backup; verify your downloaded ZIP/private dataset before ending a session.

In [ ]:
# Optional manual export after an additional completed stage:
# workflow('preserve')
print('Download:', WORK / 'weaktr-resume.zip')